# First-contact emitter assignment pipeline

This notebook runs the end-to-end demonstration requested for `LuaHistory_2026-06-23.txt`: start from the first CMO `PY_CONTACT_LOG` emission line, normalize it into the repository observation ontology, optionally ingest/query Neo4j, extract graph-neighbourhood features, produce a probabilistic assignment for the emitter platform type and nationality/country of origin, and prepare an LLM explanation-layer payload that explains—but does not change—the model probabilities.

The notebook is designed to be runnable in two modes:

1. **Offline deterministic demo** (default): uses the local parser plus a small auditable candidate-reference table derived from the first emission sensor name (`Slot Back [N-010 Zhuk-M]`). This lets the whole flow run without Neo4j credentials.
2. **Neo4j-backed run** (optional): set `USE_NEO4J = True` and provide credentials to populate/query the evidence graph with the repository modules.


In [ ]:
from __future__ import annotations

import json
import math
import os
import re
from dataclasses import asdict
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "LuaHistory_2026-06-23.txt").exists() and (REPO_ROOT.parent / "LuaHistory_2026-06-23.txt").exists():
    REPO_ROOT = REPO_ROOT.parent

LUA_HISTORY = REPO_ROOT / "LuaHistory_2026-06-23.txt"
WORK_DIR = REPO_ROOT / "notebooks" / "outputs" / "first_contact_emitter_assignment"
WORK_DIR.mkdir(parents=True, exist_ok=True)

print(f"Repository: {REPO_ROOT}")
print(f"Input log:  {LUA_HISTORY}")
print(f"Outputs:    {WORK_DIR}")


## 1. Read the first contact emission

The first line in the provided Lua history file is the first contact emission record. We keep its source line number for provenance and downstream graph audit IDs.


In [ ]:
from combat_id_calibration.cmo_observation_ingest import parse_observation_line, write_observations_jsonl

first_line_number = None
first_line = None
with LUA_HISTORY.open(encoding="utf-8", errors="replace") as handle:
    for line_number, line in enumerate(handle, start=1):
        if "PY_CONTACT_LOG" in line:
            first_line_number = line_number
            first_line = line.strip()
            break

if first_line is None:
    raise RuntimeError(f"No PY_CONTACT_LOG line found in {LUA_HISTORY}")

observation = parse_observation_line(first_line, source_line=first_line_number)
if observation is None:
    raise RuntimeError("The first PY_CONTACT_LOG line could not be parsed")

observation_record = asdict(observation)
observations_jsonl = WORK_DIR / "first_contact_observation.jsonl"
write_observations_jsonl([observation], observations_jsonl)

print(f"First emission source line: {first_line_number}")
print(json.dumps(observation_record, indent=2, sort_keys=True))
print(f"Wrote {observations_jsonl.relative_to(REPO_ROOT)}")


## 2. Optional graph ingestion

Set `USE_NEO4J = True` when a Neo4j service is available. The offline path still creates the same JSONL artifact and then performs deterministic feature extraction locally so the notebook remains reproducible in a clean development environment.


In [ ]:
USE_NEO4J = False  # Change to True to write/read an actual Neo4j evidence graph.
NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "")
NEO4J_DATABASE = os.getenv("NEO4J_DATABASE") or None

if USE_NEO4J:
    from combat_id_calibration.cmo_observation_ingest import populate_observations_neo4j
    populate_observations_neo4j([observation], NEO4J_URI, NEO4J_USER, NEO4J_PASSWORD, NEO4J_DATABASE)
    print("Observation ingested into Neo4j")
else:
    print("Offline mode: skipped Neo4j write. JSONL observation artifact is available for later ingestion.")


## 3. Candidate emitter-platform hypotheses

For the first contact, the detected emitter is `Slot Back [N-010 Zhuk-M]`. The candidates below are intentionally explicit and auditable; replace or extend them with graph-derived candidates when the reference knowledge graph contains richer radar/platform/nationality evidence.


In [ ]:
candidate_reference = [
    {
        "hypothesis": "MiG-29SMT",
        "country_of_origin": "Russia",
        "emitter_aliases": ["Slot Back", "N-010 Zhuk-M", "Zhuk-M"],
        "platform_class": "Type: Multirole (Fighter/Attack)",
        "typical_speed_kt": (250, 850),
        "typical_altitude_m": (0, 18000),
    },
    {
        "hypothesis": "MiG-29K",
        "country_of_origin": "Russia",
        "emitter_aliases": ["Slot Back", "N-010 Zhuk-M", "Zhuk-ME"],
        "platform_class": "Type: Multirole (Fighter/Attack)",
        "typical_speed_kt": (220, 850),
        "typical_altitude_m": (0, 17500),
    },
    {
        "hypothesis": "Su-30MK",
        "country_of_origin": "Russia",
        "emitter_aliases": ["Bars", "N011M"],
        "platform_class": "Type: Multirole (Fighter/Attack)",
        "typical_speed_kt": (220, 900),
        "typical_altitude_m": (0, 18000),
    },
    {
        "hypothesis": "F-16C",
        "country_of_origin": "United States",
        "emitter_aliases": ["APG-68", "AN/APG-68"],
        "platform_class": "Type: Multirole (Fighter/Attack)",
        "typical_speed_kt": (220, 900),
        "typical_altitude_m": (0, 15000),
    },
]

print(json.dumps(candidate_reference, indent=2))


## 4. Feature extraction

The repository feature contract is one row per `(scenario_id, contact_id, observation_time, hypothesis)`. In Neo4j mode, you can call `extract_features_neo4j`; in offline mode, the helper below mirrors that contract with transparent matching features from the first emission line.


In [ ]:
from combat_id_calibration.feature_extraction import ContactHypothesisFeatures, evidence_query_id, feature_logit


def _alias_match_score(sensor_name: str, aliases: list[str]) -> float:
    sensor = sensor_name.lower()
    matches = sum(1 for alias in aliases if alias.lower() in sensor)
    return min(1.0, matches / max(1, min(2, len(aliases))))


def _range_score(value: float | None, low: float, high: float) -> float:
    if value is None:
        return 0.0
    return 1.0 if low <= float(value) <= high else 0.2


def offline_features_for_candidate(obs, candidate: dict[str, object]) -> dict[str, object]:
    emission_match = _alias_match_score(obs.emission_sensor_name, candidate["emitter_aliases"])
    class_match = 1.0 if obs.emission_target_type == candidate["platform_class"] else 0.0
    speed_score = _range_score(obs.emission_speed, *candidate["typical_speed_kt"])
    altitude_score = _range_score(obs.emission_altitude, *candidate["typical_altitude_m"])
    kinematic_match = (speed_score + altitude_score) / 2.0
    supporting_paths = (2.0 * emission_match) + class_match + kinematic_match
    contradicting_paths = 0.0 if emission_match else 1.0
    contradiction_score = contradicting_paths / max(1.0, supporting_paths + contradicting_paths)
    request_ids = {
        "scenario_id": "LuaHistory_2026-06-23_first_contact",
        "contact_id": obs.observation_id,
        "observation_time": str(obs.time),
        "hypothesis": candidate["hypothesis"],
    }
    features = ContactHypothesisFeatures(
        **request_ids,
        supporting_path_count=supporting_paths,
        contradicting_path_count=contradicting_paths,
        mean_source_reliability=0.75,
        recency=1.0 / (1.0 + float(obs.emission_age or 0.0)),
        shortest_path_to_platform_class=1.0 if class_match else 3.0,
        emission_match_score=emission_match,
        kinematic_match_score=kinematic_match,
        contradiction_score=contradiction_score,
        evidence_query_id=evidence_query_id(**request_ids),
    )
    record = features.to_record()
    record["country_of_origin"] = candidate["country_of_origin"]
    record["feature_logit"] = feature_logit(features)
    record["evidence_summary"] = {
        "sensor_name": obs.emission_sensor_name,
        "matched_aliases": [a for a in candidate["emitter_aliases"] if a.lower() in obs.emission_sensor_name.lower()],
        "class_match": bool(class_match),
        "speed_kt": obs.emission_speed,
        "altitude_m": obs.emission_altitude,
    }
    return record

feature_rows = [offline_features_for_candidate(observation, candidate) for candidate in candidate_reference]
features_jsonl = WORK_DIR / "first_contact_feature_rows.jsonl"
features_jsonl.write_text("".join(json.dumps(row, sort_keys=True) + "\n" for row in feature_rows), encoding="utf-8")

print(json.dumps(feature_rows, indent=2))
print(f"Wrote {features_jsonl.relative_to(REPO_ROOT)}")


## 5. Probabilistic platform and nationality assignment

The probability model groups feature rows for the same contact/time, converts each row to a logit, applies a softmax baseline (or a fitted temperature calibrator if provided), and marginalizes platform probabilities into country/nationality probabilities.


In [ ]:
from combat_id_calibration.probability_model import run_probability_model

probabilities_jsonl = WORK_DIR / "first_contact_probability_assignment.jsonl"
assignments = run_probability_model(features_jsonl, probabilities_jsonl)
assignment = assignments[0]

print(json.dumps(assignment, indent=2))
print(f"Wrote {probabilities_jsonl.relative_to(REPO_ROOT)}")


## 6. LLM explanation-layer payload

This step uses the repository LLM explanation layer. It prepares a deterministic explanation payload and prompt from the probability record plus evidence summaries. The LLM layer is intentionally downstream of the probability model: it explains supplied probabilities and evidence without recalculating or changing them.


In [ ]:
from combat_id_calibration.llm_explainer import build_explanation_payload

best_platform = assignment["top_platform"]
supporting_evidence = []
contradicting_evidence = []
missing_evidence = [
    "Run the same contact against a populated Neo4j reference graph for independent radar/platform evidence paths.",
    "Add calibrated training data before using the softmax baseline operationally.",
    "Collect follow-up emissions, IFF, bearing/range history, and track-quality evidence to break ties between close MiG-29 variants.",
]

for row in feature_rows:
    summary = row["evidence_summary"]
    evidence_item = {
        "text": (
            f"{row['hypothesis']} matched aliases {summary['matched_aliases']} from "
            f"sensor {summary['sensor_name']}; class_match={summary['class_match']}; "
            f"speed_kt={summary['speed_kt']}; altitude_m={summary['altitude_m']}"
        ),
        "evidence_query_id": row["evidence_query_id"],
    }
    if row["hypothesis"] == best_platform or summary["matched_aliases"]:
        supporting_evidence.append(evidence_item)
    else:
        contradicting_evidence.append(evidence_item)

explanation_payload = build_explanation_payload(
    assignment,
    {
        "supporting_evidence": supporting_evidence,
        "contradicting_evidence": contradicting_evidence,
        "missing_evidence": missing_evidence,
    },
)

explanations_jsonl = WORK_DIR / "first_contact_explanation_payload.jsonl"
explanations_jsonl.write_text(json.dumps(explanation_payload, sort_keys=True) + "\n", encoding="utf-8")

print(json.dumps(explanation_payload, indent=2))
print(f"Wrote {explanations_jsonl.relative_to(REPO_ROOT)}")


## 7. Human-readable conclusion

The result below should be treated as a development baseline unless a fitted calibration model and richer Neo4j evidence graph are supplied. The notebook preserves the candidate distribution and explanation payload rather than only the winning label.


In [ ]:
print(
    f"Top emitter platform type: {assignment['top_platform']} "
    f"({assignment['top_platform_probability']:.1%})"
)
print(
    f"Top nationality / country of origin: {assignment['top_country_of_origin']} "
    f"({assignment['top_country_probability']:.1%})"
)
print("\nPlatform distribution:")
for platform, probability in assignment["platform_probabilities"].items():
    print(f"  - {platform}: {probability:.1%}")
print("\nCountry distribution:")
for country, probability in assignment["country_probabilities"].items():
    print(f"  - {country}: {probability:.1%}")
